In [0]:
df=spark.table('workspace.default.extradata')
display(df)
     

In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c + "_nulls")
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.functions import col

df = spark.table("workspace.default.extradata")

df = df.dropna(how="all")

In [0]:
df = df.filter(col("Geography").isNotNull() & col("Amount").isNotNull())

In [0]:
df = df.toDF(*[c.replace(" ", "_").lower() for c in df.columns])

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("geography").orderBy(col("amount").desc())

df_ranked = df.withColumn("rank", rank().over(windowSpec))

df_ranked.show()

In [0]:
from pyspark.sql.functions import col, lit, when, concat

# Select column
df.select(col("Amount")).show()

# Add new column
df1 = df.withColumn("new_col", lit(100))
df1.show()

# Add category column
df2 = df.withColumn(
    "Category",
    when(col("Amount") > 1000, "High").otherwise("Low")
)
df2.show()

# Concatenate columns
df3 = df.withColumn(
    "full",
    concat(col("Product"), col("Geography"))
)
df3.show()

In [0]:
df=spark.sql("SHOW TABLES IN workspace.default").show()


In [0]:
df = spark.table("workspace.default.extradata")

df.printSchema()
df.show(5)

In [0]:
from pyspark.sql.functions import trim, col

df_clean = df.select(
    trim(col("Sales Person")).alias("sales_person"),
    trim(col("Geography")).alias("geography"),
    trim(col("Product")).alias("product"),
    col("Date"),
    col("Amount").cast("int"),
    col("Customers").cast("int"),
    col("Boxes").cast("int")
)
df_clean.show()

In [0]:
from pyspark.sql.functions import to_date

df_clean = df_clean.withColumn("date", to_date("Date", "dd-MMM-yy"))
df_clean.show()


In [0]:
from pyspark.sql.functions import sum, when

df_clean.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clean.columns
]).show()


In [0]:
df_clean.filter(col("Amount") < 0).show()

In [0]:
df_clean.groupBy("geography") \
    .sum("Amount") \
    .withColumnRenamed("sum(Amount)", "total_sales") \
    .show()


In [0]:
df_clean.groupBy("product") \
    .sum("Amount") \
    .orderBy(col("sum(Amount)").desc()) \
    .show()


In [0]:
df = spark.table("workspace.default.extradata")

df = df.cache()   # 🔥 important
df.count()     

In [0]:
df.createOrReplaceTempView("clean_data")

spark.sql("""
SELECT *,
       RANK() OVER (PARTITION BY geography ORDER BY amount DESC) AS rank
FROM clean_data
""").show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col



# df=spark.table('workspace.default.extradata')
# print(df.columns)
# df.show(5)
df = spark.table("workspace.default.extradata")

df = df.toDF(*[c.replace(" ", "_").lower() for c in df.columns])
# Rename columns safely
df = df.toDF("sales_person","geography","product","date","amount","customers","boxes")

windowSpec = Window.partitionBy("geography").orderBy(col("amount").desc())

df_ranked = df.withColumn("rank", rank().over(windowSpec))

df_ranked.show()


In [0]:
data = [
    ("India", 100),
    ("India", 200),
    ("USA", 150)
]

test_df = spark.createDataFrame(data, ["geography", "amount"])

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("geography").orderBy(col("amount").desc())

test_df.withColumn("rank", rank().over(windowSpec)).show()

In [0]:
df = spark.table("workspace.default.extradata")

# 🔥 THIS LINE FIXES YOUR ISSUE
df = spark.createDataFrame(df.collect())

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("geography").orderBy(col("amount").desc())

df_ranked = df.withColumn("rank", rank().over(windowSpec))

df_ranked.show()

In [0]:
df_clean.write.mode("overwrite").saveAsTable("workspace.default.extradata_clean")

In [0]:
df = spark.table("workspace.default.extradata_clean")

df.withColumn("rank",
    rank().over(Window.partitionBy("geography").orderBy(col("amount").desc()))
).show()

In [0]:
from pyspark.sql.functions import sum

windowSpec = Window.partitionBy("geography").orderBy("date")

df_running = df_clean.withColumn(
    "running_total",
    sum("Amount").over(windowSpec)
)
df_running.show()


In [0]:
from pyspark.sql.functions import when

df_scd1 = df_clean.withColumn(
    "product",
    when(col("product") == "Mint Chip Choco", "Mint Chocolate").otherwise(col("product"))
)
df_scd1.show()

In [0]:
df_scd1.write.mode("overwrite").saveAsTable("workspace.default.extradata_scd1")


In [0]:
from pyspark.sql.functions import lit, current_date

df_scd2 = df_clean.withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None)) \
    .withColumn("is_current", lit(True))


In [0]:
df = spark.read.table("workspace.default.extradata")
df.show()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.sales_delta")


In [0]:
df_clean = df.toDF(
    "sales_person",
    "geography",
    "product",
    "date",
    "amount",
    "customers",
    "boxes"
)

df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.sales_delta")

In [0]:
df.write.format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.sales_delta")


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "workspace.default.sales_delta")

deltaTable.alias("target").merge(
    df_clean.alias("source"),
    "target.sales_person = source.sales_person AND target.date = source.date"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


In [0]:
from pyspark.sql.functions import col, to_date
df = spark.table("workspace.default.extradata")
df_clean = df.select(
    col("Sales Person").alias("sales_person"),
    col("Geography").alias("geography"),
    col("Product").alias("product"),
    to_date(col("Date"), "dd-MMM-yy").alias("date"),
    col("Amount").alias("amount"),
    col("Customers").alias("customers"),
    col("Boxes").alias("boxes")
)


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "workspace.default.sales_delta")

deltaTable.alias("target").merge(
    df_clean.alias("source"),
    "target.sales_person = source.sales_person AND target.date = source.date"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


In [0]:
print("SOURCE SCHEMA")
df_clean.printSchema()

print("TARGET SCHEMA")
spark.table("workspace.default.sales_delta").printSchema()


In [0]:
# Step 1: Load
df = spark.table("workspace.default.extradata")
df.printSchema()
# Step 2: Clean
from pyspark.sql.functions import col, to_date

df_clean = df.select(
    col("Sales Person").alias("sales_person"),
    col("Geography").alias("geography"),
    col("Product").alias("product"),
    to_date(col("Date"), "dd-MMM-yy").alias("date"),
    col("Amount").alias("amount"),
    col("Customers").alias("customers"),
    col("Boxes").alias("boxes")
)

# Step 3: Reset table
spark.sql("DROP TABLE IF EXISTS workspace.default.sales_delta")

# Step 4: Create table
df_clean.write.format("delta").saveAsTable("workspace.default.sales_delta")

# Step 5: MERGE
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "workspace.default.sales_delta")

deltaTable.alias("target").merge(
    df_clean.alias("source"),
    "target.sales_person = source.sales_person AND target.date = source.date"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
